# Notebook Task — Join Customers and Credit Applications

**Objective:** Combine staged credit applications with customer names and states.

## Why?

**For our project:** Prepare application records for state-based reporting.

**For Data Engineering:** Make the reporting task wait for both ingestion inputs and provide one reusable dataset to the For each task.

#### 01 — Configure Notebook Parameters

Read the catalog and schema parameters to identify the tables used by this task.

In [ ]:
# Define defaults for manual execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")

# Read the current parameter values.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

#### 02 — Build the Joined Application Dataset

Join staged applications to customer details using `customer_id` and save the result as a Delta table for state-based reporting.

In [ ]:
# Enrich applications with customer names and states.
applications_df = spark.sql(f"""
    SELECT
        a.application_id,
        c.customer_id,
        c.customer_name,
        c.state,
        a.application_date,
        a.requested_amount,
        a.status
    FROM {namespace}.applications_staging AS a
    INNER JOIN {namespace}.customers_staging AS c
        ON a.customer_id = c.customer_id
""")

# Persist the joined application records.
target_table = f"{namespace}.customers_applications_silver"

applications_df.write.format("delta").mode("overwrite").saveAsTable(
    target_table
)

# Display the persisted result for validation.
stored_df = spark.table(target_table)

print(f"Application records: {stored_df.count()}")
display(stored_df.orderBy("application_id"))

Application records: 6


application_id,customer_id,customer_name,state,application_date,requested_amount,status
2001,101,Ana Ribeiro,PE,2026-10-06,3500.00,APPROVED
2002,102,Carlos Lima,SP,2026-10-07,8200.00,PENDING
2003,103,Maria Costa,RJ,2026-10-08,5000.00,REJECTED
2004,104,Pedro Alves,MG,2026-10-09,12000.00,APPROVED
2005,105,Julia Santos,BA,2026-10-10,2600.00,APPROVED
2006,101,Ana Ribeiro,PE,2026-10-11,4500.00,PENDING
